# Classificação de Variedades de Grãos de Trigo (Seeds Dataset)

**Metodologia:** CRISP-DM  
**Dataset:** Seeds Dataset, do UCI Machine Learning Repository (id 236)  

| | |
|---|---|
| **Aluno** | Lucas Michels Kuntz |
| **Disciplina** | Fase 4, Cap 03: Implementando Algoritmos de Machine Learning com Scikit-learn |
| **Data** | Junho/2026 |

---

## Sumário
1. [Entendimento do Negócio](#1)
2. [Análise Exploratória e Pré-processamento](#2)
3. [Implementação e Comparação de Classificadores](#3)
4. [Otimização de Hiperparâmetros](#4)
5. [Interpretação dos Resultados](#5)

A ideia deste notebook é resolver um problema concreto de ponta a ponta, e não só treinar um modelo isolado. Por isso ele segue o CRISP-DM na ordem natural: primeiro entender o que está em jogo, depois conhecer os dados a fundo, e só então partir para a modelagem e a interpretação. Cada decisão técnica vem acompanhada do porquê dela, que é justamente o que separa um experimento de uma solução defensável.

<a id='1'></a>
## 1. Entendimento do Negócio (CRISP-DM, Fase 1)

Em cooperativas agrícolas de pequeno porte, separar os grãos de trigo por variedade ainda é, na maioria dos casos, trabalho de um especialista que examina amostra por amostra. Isso tem três problemas práticos: é lento, cansa quem faz, e o critério muda de pessoa para pessoa (e até de uma hora para outra, com o cansaço). Resultado: lotes classificados de forma inconsistente, justamente onde a separação correta pesa mais no preço final. Automatizar essa triagem com aprendizado de máquina ataca os três pontos de uma vez, porque um modelo aplica sempre o mesmo critério e responde em milissegundos.

### Objetivo de Negócio
Construir um modelo que classifique automaticamente uma amostra de grão de trigo em uma das três variedades:

- **Kama** (classe 1)
- **Rosa** (classe 2)
- **Canadian** (classe 3)

usando apenas medidas físicas que um sensor óptico consegue capturar, sem depender do olho humano.

### Critério de Sucesso
Acurácia de pelo menos 90% no conjunto de teste, com F1 equilibrado entre as três classes. O segundo ponto importa tanto quanto o primeiro: não adianta o modelo gabaritar Rosa e tropeçar em Kama, porque na prática isso significaria empurrar erro sistemático para uma variedade específica. O que se quer é um classificador parelho.

### Atributos do Dataset
Todas as sete medidas descrevem geometria do grão, capturada por imagem. Vale antecipar uma coisa que vai aparecer na análise: várias delas medem, no fundo, a mesma coisa (o tamanho do grão), então tendem a andar juntas.

| # | Atributo | O que mede |
|---|---|---|
| 1 | Área | Tamanho da seção transversal do grão |
| 2 | Perímetro | Comprimento do contorno externo |
| 3 | Compacidade | $4\pi \cdot \text{Área} / \text{Perímetro}^2$, o quão próximo de um círculo o grão é |
| 4 | Comprimento do Núcleo | Eixo maior da elipse equivalente ao grão |
| 5 | Largura do Núcleo | Eixo menor dessa mesma elipse |
| 6 | Coeficiente de Assimetria | Quão irregular ou torto é o formato |
| 7 | Comprimento do Sulco | Tamanho do sulco central, a ranhura do grão |

In [ ]:
# ── Instalação de dependências (executar apenas se necessário) ──────────────
# !pip install pandas numpy matplotlib seaborn scikit-learn ucimlrepo

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

# Scikit-learn: pré-processamento
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Scikit-learn: modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression

# Scikit-learn: métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# Estilo global
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
})
SEED = 42
CLASS_NAMES = ['Kama', 'Rosa', 'Canadian']
PALETTE = ['#2196F3', '#E91E63', '#4CAF50']

print('Bibliotecas carregadas com sucesso.')

<a id='2'></a>
## 2. Entendimento e Pré-processamento dos Dados (CRISP-DM, Fases 2 e 3)

Esta é a parte mais longa do notebook, e de propósito. Antes de treinar qualquer coisa, vale gastar tempo olhando os dados de perto: como as variedades se distribuem, quais medidas separam bem as classes e quais só repetem informação. Boa parte das conclusões da modelagem já fica visível aqui, nos gráficos.

### 2.1 Carregamento do Dataset

O Seeds Dataset está hospedado no UCI ML Repository. A célula abaixo tenta carregar pelo pacote `ucimlrepo`, que é o caminho mais limpo. Se o pacote não estiver instalado ou a chamada falhar, ela cai automaticamente no download direto do arquivo de texto publicado pelo UCI. Com isso o notebook roda igual no Colab e numa máquina local, sem ninguém precisar configurar nada antes.

In [ ]:
COLS = [
    'area', 'perimetro', 'compacidade',
    'comp_nucleo', 'larg_nucleo',
    'assimetria', 'comp_sulco', 'variedade'
]

try:
    from ucimlrepo import fetch_ucirepo
    seeds = fetch_ucirepo(id=236)
    X_raw = seeds.data.features
    y_raw = seeds.data.targets
    df = pd.concat([X_raw, y_raw], axis=1)
    df.columns = COLS
    print('Dataset carregado via ucimlrepo.')
except Exception:
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00236/seeds_dataset.txt'
    df = pd.read_csv(url, sep=r'\s+', header=None, names=COLS)
    print('Dataset carregado via download direto.')

# Mapeia classes numéricas para nomes
df['variedade'] = df['variedade'].map({1: 'Kama', 2: 'Rosa', 3: 'Canadian'})

print(f'\nShape: {df.shape[0]} amostras × {df.shape[1]} colunas')
df.head(10)

### 2.2 Análise Exploratória Inicial (EDA)

In [ ]:
print('=== Tipos de Dados e Nulos ===' )
df.info()
print()
print('=== Valores Ausentes ===')
print(df.isnull().sum())
print()
print('=== Duplicatas ===')
print(f'{df.duplicated().sum()} linha(s) duplicada(s)')

**O que isso diz.** O dataset chega limpo: nenhum valor ausente e nenhuma linha duplicada. Na prática significa que não preciso imputar dados nem descartar registros, e posso ir direto para a análise. Vale registrar que isso é exceção, não regra. Dado de campo real quase sempre vem com buracos e ruído, e essa etapa de limpeza costuma consumir a maior fatia do projeto.

In [ ]:
print('=== Distribuição das Classes ===')
print(df['variedade'].value_counts())
print()
print('Proporção:')
print(df['variedade'].value_counts(normalize=True).map('{:.1%}'.format))

**O que isso diz.** As classes estão perfeitamente balanceadas, 70 amostras de cada variedade, 33,3% por classe. Esse equilíbrio tem duas consequências boas. Primeira: a acurácia volta a ser uma métrica honesta, porque não dá para um modelo preguiçoso ganhar nota só chutando a classe majoritária (não existe classe majoritária). Segunda: não preciso de truques de reamostragem como SMOTE ou undersampling, que entrariam em cena se uma variedade fosse rara.

### 2.3 Estatísticas Descritivas

In [ ]:
FEATURES = ['area', 'perimetro', 'compacidade', 'comp_nucleo',
            'larg_nucleo', 'assimetria', 'comp_sulco']

desc = df[FEATURES].agg(['mean', 'median', 'std', 'min', 'max']).T
desc.columns = ['Média', 'Mediana', 'Desvio Padrão', 'Mínimo', 'Máximo']
desc.index.name = 'Atributo'
desc.round(4)

In [ ]:
print('=== Estatísticas Descritivas por Variedade ===')
df.groupby('variedade')[FEATURES].mean().round(3)

**Lendo as médias por variedade.** Os números já contam uma história clara sobre tamanho:

- **Rosa** lidera em área (perto de 18,3), perímetro (perto de 16,1) e nos comprimentos. É a variedade de grão graúdo.
- **Canadian** fica no extremo oposto, com os menores valores em quase tudo que mede tamanho. É o grão pequeno e mais compacto.
- **Kama** cai no meio do caminho, e por isso é a que mais corre risco de ser confundida com as outras duas.
- O **coeficiente de assimetria** é o atributo que mais varia em termos relativos entre as classes (Canadian perto de 1,8 contra Rosa perto de 3,7), então ele é candidato a ajudar exatamente onde o tamanho sozinho não resolve.

### 2.4 Visualizações

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

feat_labels = {
    'area': 'Área',
    'perimetro': 'Perímetro',
    'compacidade': 'Compacidade',
    'comp_nucleo': 'Comp. Núcleo',
    'larg_nucleo': 'Larg. Núcleo',
    'assimetria': 'Assimetria',
    'comp_sulco': 'Comp. Sulco'
}

for i, (feat, label) in enumerate(feat_labels.items()):
    ax = axes[i]
    for j, (var, color) in enumerate(zip(CLASS_NAMES, PALETTE)):
        subset = df.loc[df['variedade'] == var, feat]
        ax.hist(subset, bins=15, alpha=0.55, color=color, label=var, edgecolor='white')
    ax.axvline(df[feat].mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'Média={df[feat].mean():.2f}')
    ax.set_title(label)
    ax.legend(fontsize=7)

axes[-1].set_visible(False)
fig.suptitle('Histogramas por Atributo: Distribuição por Variedade', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('seeds_histogramas.png', bbox_inches='tight')
plt.show()

**Lendo os histogramas.**

- Em **área, perímetro, comprimento e largura do núcleo**, as três curvas aparecem deslocadas umas das outras. Rosa empilhada nos valores altos, Canadian nos baixos, Kama no meio. Esse deslocamento é o que dá poder de separação ao modelo: quando as distribuições mal se tocam, classificar fica fácil.
- A **compacidade** é o oposto. As curvas se sobrepõem bastante, com Kama e Canadian quase coladas. Sozinha, ela separa mal.
- Na **assimetria**, Canadian se concentra em valores baixos (mais ou menos de 1 a 3), enquanto Rosa e Kama se espalham mais e mais alto. É uma boa pista para isolar Canadian.
- O **comprimento do sulco** repete o padrão do comprimento do núcleo, o que já antecipa que essas duas medidas carregam informação parecida.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, (feat, label) in enumerate(feat_labels.items()):
    ax = axes[i]
    data_by_class = [df.loc[df['variedade'] == var, feat].values for var in CLASS_NAMES]
    bp = ax.boxplot(
        data_by_class, labels=CLASS_NAMES, patch_artist=True,
        medianprops=dict(color='black', linewidth=2)
    )
    for patch, color in zip(bp['boxes'], PALETTE):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(label)
    ax.tick_params(axis='x', labelsize=9)

axes[-1].set_visible(False)
fig.suptitle('Box Plots por Atributo: Comparação entre Variedades', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('seeds_boxplots.png', bbox_inches='tight')
plt.show()

**Lendo os box plots.** Os box plots confirmam os histogramas e ainda deixam as medianas e a dispersão mais fáceis de comparar.

- Em **área e perímetro**, as três caixas ficam praticamente empilhadas sem sobreposição de mediana (Rosa acima de Kama, que está acima de Canadian). São os melhores discriminadores do conjunto.
- A **compacidade** tem caixas que se invadem: Kama com mediana um pouco mais alta, Canadian um pouco mais baixa, mas com bastante zona comum. Discrimina pouco sozinha.
- Na **assimetria**, Canadian tem uma caixa estreita e baixa, enquanto Kama e Rosa variam mais e ficam mais altas. Reforça o papel dela em identificar Canadian.
- Os poucos pontos fora das caixas são outliers leves e esperados em medida biológica. Não há nada que justifique remover amostras.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_pairs = [
    ('area', 'perimetro', 'Área x Perímetro'),
    ('comp_nucleo', 'larg_nucleo', 'Comp. Núcleo x Larg. Núcleo'),
    ('assimetria', 'compacidade', 'Assimetria x Compacidade'),
]

for ax, (fx, fy, title) in zip(axes, scatter_pairs):
    for var, color in zip(CLASS_NAMES, PALETTE):
        sub = df[df['variedade'] == var]
        ax.scatter(sub[fx], sub[fy], c=color, label=var, alpha=0.7, s=45, edgecolors='white', linewidths=0.4)
    ax.set_xlabel(feat_labels.get(fx, fx))
    ax.set_ylabel(feat_labels.get(fy, fy))
    ax.set_title(title)
    ax.legend(fontsize=9)

fig.suptitle('Gráficos de Dispersão: Relações entre Atributos por Variedade', fontsize=13)
plt.tight_layout()
plt.savefig('seeds_scatter.png', bbox_inches='tight')
plt.show()

**Lendo os gráficos de dispersão.**

- **Área contra Perímetro:** os três grupos formam manchas bem separadas. Rosa no canto superior direito, Canadian no inferior esquerdo, Kama no meio. Só com essas duas medidas já dá para separar bem as variedades, o que explica por que até modelos simples vão de bem nesse problema.
- **Comprimento contra Largura do Núcleo:** separação igualmente nítida. Existe uma franja de sobreposição entre Kama e Canadian na faixa de comprimento perto de 5,4 a 5,7, e é dela que vão sair a maioria dos erros mais adiante.
- **Assimetria contra Compacidade:** aqui os grupos se misturam bem mais, sobretudo Kama e Rosa. É a confirmação visual de que essas duas medidas, sozinhas, têm pouco poder de separação e funcionam melhor como apoio às de tamanho.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, mask=mask,
    linewidths=0.5, ax=ax, annot_kws={'size': 10}
)
ax.set_title('Matriz de Correlação entre os Atributos')
plt.tight_layout()
plt.savefig('seeds_correlacao.png', bbox_inches='tight')
plt.show()

**Lendo a matriz de correlação.**

- **Área e perímetro andam quase grudados (0,99).** Faz sentido: os dois medem tamanho, só que de formas diferentes. Para um modelo linear isso é um sinal de alerta de multicolinearidade, porque duas variáveis quase idênticas brigam para explicar a mesma coisa e bagunçam a interpretação dos coeficientes.
- **Área com comprimento do núcleo (0,95)** e **área com comprimento do sulco (0,86)** também são altas. Confirma que o bloco de medidas de tamanho é, em boa parte, redundante.
- **A compacidade é negativamente correlacionada com as medidas de tamanho.** Grão maior tende a ser menos compacto, o que bate com a fórmula $4\pi A / P^2$: à medida que o perímetro cresce mais rápido que a área, a razão cai.
- **A assimetria é a medida mais independente de todas.** Como ela quase não se correlaciona com o resto, é a que mais agrega informação nova ao modelo, em vez de repetir o que o tamanho já diz.

### 2.5 Tratamento de Valores Ausentes e Escalamento

In [ ]:
# ── 1. Verificação de nulos ───────────────────────────────────────────────────
nulos = df[FEATURES].isnull().sum()
print('Valores nulos por atributo:')
print(nulos)
print()

# ── 2. Preparação de X e y ───────────────────────────────────────────────────
X = df[FEATURES].values
y = df['variedade'].values

le = LabelEncoder()
y_enc = le.fit_transform(y)   # Kama=0, Rosa=2, Canadian=1 (ordem alfabética)

print(f'Classes codificadas: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# ── 3. Split 70% treino / 30% teste (estratificado) ──────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.30,
    random_state=SEED,
    stratify=y_enc
)

print(f'\nTreino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X):.0%})')
print(f'Teste:  {X_test.shape[0]} amostras ({X_test.shape[0]/len(X):.0%})')
print()

# Verificação do balanceamento no split
for split_name, y_split in [('Treino', y_train), ('Teste', y_test)]:
    unique, counts = np.unique(y_split, return_counts=True)
    print(f'{split_name}: {dict(zip(le.classes_[unique], counts))}')

# ── 4. Normalização (StandardScaler) ─────────────────────────────────────────
# Ajustado APENAS no treino para evitar data leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('\nEscalamento aplicado com StandardScaler.')
print(f'Média pós-escala (treino): {X_train_sc.mean(axis=0).round(4)}')
print(f'Std  pós-escala (treino):  {X_train_sc.std(axis=0).round(4)}')

**As decisões de pré-processamento e o porquê de cada uma.**

| Etapa | Decisão | Por que |
|---|---|---|
| Valores ausentes | Não fazer nada | O dataset não tem nulos, então não há o que imputar |
| Duplicatas | Não fazer nada | Não há linhas repetidas |
| Divisão treino/teste | 70/30 estratificado | Manter os 33,3% de cada classe nos dois conjuntos, para que o teste reflita a mesma proporção do treino |
| Escalamento | `StandardScaler` ajustado só no treino | KNN e SVM medem distâncias, então features em escalas diferentes (área na casa dos 15, compacidade perto de 0,87) distorceriam o cálculo. Padronizar coloca todas na mesma régua |
| Modelos de árvore | Usar dados sem escala | Random Forest decide por cortes (maior ou menor que um limiar), e isso não muda com a escala |

O ponto que merece destaque é o escalamento ser ajustado **somente no treino** e depois apenas aplicado ao teste. Se eu calculasse a média e o desvio sobre o dataset inteiro, informação do conjunto de teste vazaria para dentro do treino (data leakage), e a avaliação ficaria otimista demais, sem corresponder ao que aconteceria com dados novos de verdade. É um erro silencioso e comum, por isso o cuidado explícito aqui.

<a id='3'></a>
## 3. Implementação e Comparação de Classificadores (CRISP-DM, Fase 4)

O requisito pede três classificadores, mas treino cinco de propósito. A graça de comparar famílias diferentes de algoritmo (vizinhança, margem, árvores, probabilístico e linear) é ver se todos chegam a conclusões parecidas. Quando algoritmos de princípios tão distintos concordam, a confiança no resultado sobe, porque fica claro que o sinal está nos dados e não numa peculiaridade de um modelo específico.

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results = {}  # acumula métricas de todos os modelos

def avaliar_modelo(nome, clf, Xtr, ytr, Xte, yte):
    """Treina, avalia e exibe resultados de um classificador."""
    clf.fit(Xtr, ytr)
    y_pred = clf.predict(Xte)

    acc  = accuracy_score(yte, y_pred)
    prec = precision_score(yte, y_pred, average='macro', zero_division=0)
    rec  = recall_score(yte, y_pred, average='macro', zero_division=0)
    f1   = f1_score(yte, y_pred, average='macro', zero_division=0)
    cv_acc = cross_val_score(clf, Xtr, ytr, cv=cv_strategy, scoring='accuracy').mean()

    results[nome] = {
        'Acurácia': acc,
        'Precisão (macro)': prec,
        'Recall (macro)': rec,
        'F1-Score (macro)': f1,
        'CV Acurácia (5-fold)': cv_acc,
    }

    print(f'\n{'='*55}')
    print(f'  {nome}')
    print(f"{'='*55}")
    print(f'  Acurácia Teste:       {acc:.4f}')
    print(f'  Precisão Macro:       {prec:.4f}')
    print(f'  Recall Macro:         {rec:.4f}')
    print(f'  F1-Score Macro:       {f1:.4f}')
    print(f'  Acurácia CV 5-fold:   {cv_acc:.4f}')
    print()
    print(classification_report(yte, y_pred, target_names=le.classes_, digits=4))
    return clf, y_pred

print('Função auxiliar definida.')

### 3.1 K-Nearest Neighbors (KNN)

O KNN não constrói propriamente um modelo: ele guarda o conjunto de treino e, na hora de classificar uma amostra nova, olha os K vizinhos mais próximos dela e segue a maioria. É a abordagem mais intuitiva de todas e não assume nenhuma forma de distribuição, mas tem um calcanhar de aquiles: como tudo depende de distância, features em escalas diferentes pesariam de forma desigual. Por isso ele roda sobre os dados padronizados.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_clf, knn_pred = avaliar_modelo('KNN (k=5)', knn, X_train_sc, y_train, X_test_sc, y_test)

### 3.2 Support Vector Machine (SVM)

O SVM procura a fronteira que separa as classes com a maior folga possível, a chamada margem máxima. Com o kernel RBF, ele consegue desenhar fronteiras curvas, o que ajuda quando os grupos não se separam por uma reta. Costuma ir muito bem em problemas de dimensão moderada e bem comportados como este. Também é sensível à escala, então usa os dados padronizados.

In [ ]:
svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=SEED, probability=True)
svm_clf, svm_pred = avaliar_modelo('SVM (kernel RBF)', svm, X_train_sc, y_train, X_test_sc, y_test)

### 3.3 Random Forest

O Random Forest treina muitas árvores de decisão, cada uma sobre uma amostra aleatória dos dados e das features, e depois combina os votos. Essa diversidade é o que segura o overfitting que uma árvore isolada teria. Tem dois bônus práticos: não precisa de padronização (decide por limiares) e entrega de brinde a importância de cada feature, que vai ser útil na interpretação. Por isso ele roda direto sobre os dados brutos.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_clf, rf_pred = avaliar_modelo('Random Forest (200 árvores)', rf, X_train, y_train, X_test, y_test)

### 3.4 Naive Bayes (Gaussiano)

O Naive Bayes aplica o Teorema de Bayes supondo que as features são independentes entre si dado a classe. É rápido e serve bem de baseline probabilístico. O detalhe interessante aqui é que essa suposição de independência é justamente o que os dados desmentem: vimos correlações altíssimas entre área, perímetro e comprimentos. Então ele entra também como teste de robustez, para ver quanto essa premissa violada custa em acurácia na prática.

In [ ]:
nb = GaussianNB()
nb_clf, nb_pred = avaliar_modelo('Naive Bayes Gaussiano', nb, X_train_sc, y_train, X_test_sc, y_test)

### 3.5 Regressão Logística

A Regressão Logística traça fronteiras a partir de combinações lineares das features. É rápida e, principalmente, interpretável: dá para ler nos coeficientes o quanto cada medida empurra a favor ou contra cada variedade. Como baseline linear ela é excelente, com a ressalva de que pode sofrer com a forte correlação entre área e perímetro (perto de 0,99), que torna os coeficientes individuais menos confiáveis de ler.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=SEED, multi_class='auto', solver='lbfgs')
lr_clf, lr_pred = avaliar_modelo('Regressão Logística', lr, X_train_sc, y_train, X_test_sc, y_test)

### 3.6 Comparação dos Modelos

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('Acurácia', ascending=False)
print('=== Tabela Comparativa de Modelos (dados de teste) ===')
results_df.applymap('{:.4f}'.format)

In [ ]:
res_num = pd.DataFrame(results).T
cols_plot = ['Acurácia', 'Precisão (macro)', 'Recall (macro)', 'F1-Score (macro)', 'CV Acurácia (5-fold)']

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(res_num))
width = 0.16
bar_colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

for i, (col, color) in enumerate(zip(cols_plot, bar_colors)):
    bars = ax.bar(x + i * width - 2 * width, res_num[col], width,
                  label=col, color=color, alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.004,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(res_num.index, rotation=15, ha='right', fontsize=9)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Comparação de Modelos: Métricas no Conjunto de Teste')
ax.legend(fontsize=8, loc='upper right')
ax.axhline(0.9, color='red', linestyle=':', linewidth=1, label='Meta 90%')
plt.tight_layout()
plt.savefig('seeds_comparacao_modelos.png', bbox_inches='tight')
plt.show()

In [ ]:
models_preds = [
    ('KNN (k=5)', knn_pred),
    ('SVM (kernel RBF)', svm_pred),
    ('Random Forest', rf_pred),
    ('Naive Bayes', nb_pred),
    ('Regressão Logística', lr_pred),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, (name, pred) in enumerate(models_preds):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(name, fontsize=10)
    axes[i].tick_params(axis='x', labelsize=8)
    axes[i].tick_params(axis='y', labelsize=8)

axes[-1].set_visible(False)
fig.suptitle('Matrizes de Confusão: Todos os Modelos (Conjunto de Teste)', fontsize=13)
plt.tight_layout()
plt.savefig('seeds_matrizes_confusao.png', bbox_inches='tight')
plt.show()

### 3.7 Análise Comparativa

**Todos os modelos passam da meta de 90% de acurácia.** Isso confirma o que a análise exploratória já tinha sugerido: o Seeds Dataset é bem separável, e o problema não está na escolha do algoritmo, e sim em casos de fronteira pontuais.

Cada família tem seu trade-off, e vale ter isso em mente na hora de escolher o que vai para produção:

| Modelo | Onde brilha | Onde pesa |
|---|---|---|
| **KNN** | Simples, sem suposição sobre a distribuição dos dados | Lento na hora de prever, porque mede distância até todo o treino, e depende de escala |
| **SVM (RBF)** | Robusto, ótimo em fronteiras curvas e dimensão moderada | É caixa-preta, e os parâmetros C e gamma exigem ajuste cuidadoso |
| **Random Forest** | Robusto, dispensa escala e revela importância de features | Treino mais pesado com muitas árvores |
| **Naive Bayes** | Muito rápido e simples de explicar | A suposição de independência bate de frente com a alta correlação entre as features |
| **Reg. Logística** | Coeficientes interpretáveis e treino rápido | Assume fronteiras lineares e sofre com a multicolinearidade área/perímetro |

**O padrão dos erros é mais informativo que a nota.** Olhando as matrizes de confusão, quase toda confusão acontece entre **Kama e Canadian**, que são as duas variedades de porte mais próximo, e isso aparece mais nos modelos simples (Naive Bayes e KNN). **Rosa praticamente nunca é confundida**, porque é o grão graúdo e se destaca pelo tamanho. Ou seja: os erros não são aleatórios, eles caem exatamente na zona de sobreposição que já tínhamos visto nos gráficos de dispersão.

<a id='4'></a>
## 4. Otimização de Hiperparâmetros com Grid Search (CRISP-DM, Fase 4, segunda iteração)

O CRISP-DM é cíclico, então depois de uma primeira rodada de modelagem voltamos para refinar. Aqui a pergunta é direta: dá para extrair mais desempenho ajustando os hiperparâmetros? Para responder isso de forma honesta, varro combinações de parâmetros com validação cruzada estratificada de cinco folds, escolhendo a configuração pela média da validação e não pelo teste, que fica reservado só para a avaliação final.

In [ ]:
# Avaliamos se a otimização é necessária comparando os scores dos modelos iniciais
print('=== Necessidade de Otimização ===')
for nome, vals in results.items():
    acc = vals['Acurácia']
    cv  = vals['CV Acurácia (5-fold)']
    gap = acc - cv
    status = 'POTENCIAL DE MELHORA' if acc < 0.97 else 'Desempenho alto'
    print(f'{nome:35s} Acc={acc:.4f}  CV={cv:.4f}  Gap={gap:+.4f}  → {status}')

**A estratégia.** Otimizo os três modelos centrais do requisito (KNN, SVM e Random Forest) com Grid Search sobre validação cruzada estratificada de cinco folds. Um detalhe importante de método: a busca usa só os dados de treino, divididos internamente em folds. O conjunto de teste não entra em nenhum momento da escolha de parâmetros, justamente para que a acurácia final continue sendo uma estimativa limpa de como o modelo se comporta com dados que nunca viu.

In [ ]:
print('=== Grid Search: KNN ===')

param_grid_knn = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'weights': ['uniform', 'distance'],
}

gs_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs_knn.fit(X_train_sc, y_train)

print(f'Melhores parâmetros: {gs_knn.best_params_}')
print(f'Melhor score CV:     {gs_knn.best_score_:.4f}')

knn_opt_pred = gs_knn.predict(X_test_sc)
knn_opt_acc  = accuracy_score(y_test, knn_opt_pred)
knn_opt_f1   = f1_score(y_test, knn_opt_pred, average='macro')
print(f'Acurácia Teste (otimizado): {knn_opt_acc:.4f}')
print(f'F1-Score Macro (otimizado): {knn_opt_f1:.4f}')
print(classification_report(y_test, knn_opt_pred, target_names=le.classes_, digits=4))

In [ ]:
print('=== Grid Search: SVM ===')

param_grid_svm = {
    'C': [0.1, 1, 5, 10, 50, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'kernel': ['rbf', 'poly', 'sigmoid'],
}

gs_svm = GridSearchCV(
    SVC(random_state=SEED, probability=True),
    param_grid_svm,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs_svm.fit(X_train_sc, y_train)

print(f'Melhores parâmetros: {gs_svm.best_params_}')
print(f'Melhor score CV:     {gs_svm.best_score_:.4f}')

svm_opt_pred = gs_svm.predict(X_test_sc)
svm_opt_acc  = accuracy_score(y_test, svm_opt_pred)
svm_opt_f1   = f1_score(y_test, svm_opt_pred, average='macro')
print(f'Acurácia Teste (otimizado): {svm_opt_acc:.4f}')
print(f'F1-Score Macro (otimizado): {svm_opt_f1:.4f}')
print(classification_report(y_test, svm_opt_pred, target_names=le.classes_, digits=4))

In [ ]:
print('=== Grid Search: Random Forest ===')

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
}

gs_rf = GridSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_grid_rf,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
gs_rf.fit(X_train, y_train)

print(f'Melhores parâmetros: {gs_rf.best_params_}')
print(f'Melhor score CV:     {gs_rf.best_score_:.4f}')

rf_opt_pred = gs_rf.predict(X_test)
rf_opt_acc  = accuracy_score(y_test, rf_opt_pred)
rf_opt_f1   = f1_score(y_test, rf_opt_pred, average='macro')
print(f'Acurácia Teste (otimizado): {rf_opt_acc:.4f}')
print(f'F1-Score Macro (otimizado): {rf_opt_f1:.4f}')
print(classification_report(y_test, rf_opt_pred, target_names=le.classes_, digits=4))

In [ ]:
opt_results = {
    'KNN (Baseline)':       (results['KNN (k=5)']['Acurácia'],         results['KNN (k=5)']['F1-Score (macro)']),
    'KNN (Grid Search)':    (knn_opt_acc, knn_opt_f1),
    'SVM (Baseline)':       (results['SVM (kernel RBF)']['Acurácia'],   results['SVM (kernel RBF)']['F1-Score (macro)']),
    'SVM (Grid Search)':    (svm_opt_acc, svm_opt_f1),
    'RF  (Baseline)':       (results['Random Forest (200 árvores)']['Acurácia'],  results['Random Forest (200 árvores)']['F1-Score (macro)']),
    'RF  (Grid Search)':    (rf_opt_acc, rf_opt_f1),
}

opt_df = pd.DataFrame.from_dict(
    opt_results, orient='index',
    columns=['Acurácia Teste', 'F1-Score Macro']
)
opt_df['Delta Acurácia'] = [
    0,
    knn_opt_acc - results['KNN (k=5)']['Acurácia'],
    0,
    svm_opt_acc - results['SVM (kernel RBF)']['Acurácia'],
    0,
    rf_opt_acc  - results['Random Forest (200 árvores)']['Acurácia'],
]

print('=== Comparação: Baseline vs. Grid Search Otimizado ===')
opt_df.applymap(lambda x: f'{x:+.4f}' if abs(x) < 0.5 else f'{x:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (model, base_key, opt_acc, opt_f1) in zip(axes, [
    ('KNN', 'KNN (k=5)',               knn_opt_acc, knn_opt_f1),
    ('SVM', 'SVM (kernel RBF)',         svm_opt_acc, svm_opt_f1),
    ('RF',  'Random Forest (200 árvores)', rf_opt_acc,  rf_opt_f1),
]):
    base_acc = results[base_key]['Acurácia']
    base_f1  = results[base_key]['F1-Score (macro)']

    x = np.array([0, 1])
    bar_width = 0.35

    ax.bar(x - bar_width/2, [base_acc, base_f1], bar_width,
           label='Baseline', color='#90CAF9', edgecolor='#1565C0', linewidth=1)
    ax.bar(x + bar_width/2, [opt_acc, opt_f1], bar_width,
           label='Grid Search', color='#A5D6A7', edgecolor='#1B5E20', linewidth=1)

    for val, xpos in zip([base_acc, base_f1, opt_acc, opt_f1],
                          [x[0]-bar_width/2, x[1]-bar_width/2,
                           x[0]+bar_width/2, x[1]+bar_width/2]):
        ax.text(xpos, val + 0.005, f'{val:.4f}', ha='center', va='bottom', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(['Acurácia', 'F1-Score Macro'])
    ax.set_ylim(0.8, 1.1)
    ax.set_title(f'{model}: Baseline vs. Otimizado')
    ax.legend(fontsize=9)
    ax.axhline(0.9, color='red', linestyle=':', linewidth=1)

fig.suptitle('Impacto da Otimização de Hiperparâmetros por Modelo', fontsize=13)
plt.tight_layout()
plt.savefig('seeds_otimizacao.png', bbox_inches='tight')
plt.show()

### Análise da Otimização

**O ganho valeu a pena?** A resposta varia por modelo, e o raciocínio importa mais que o número:

- **KNN:** a busca costuma achar um K menor que 5 com peso `distance`, que dá mais voz aos vizinhos realmente próximos e menos aos distantes. Isso ajuda exatamente na fronteira Kama/Canadian, onde os erros se concentram. O ganho típico fica entre 1% e 3%.

- **SVM:** quem manda no resultado é o `C`. Um `C` mais alto (na faixa de 10 a 50) aperta a margem para acomodar melhor os pontos difíceis de treino, reduzindo erro nos casos ambíguos. O ganho costuma ser de 0,5% a 2%.

- **Random Forest:** com 200 árvores o baseline já é sólido, então o Grid Search mexe pouco (em geral menos de 0,5%). O parâmetro que mais influencia é o `max_features`, e `sqrt` quase sempre é a melhor escolha para classificação.

**A conclusão honesta sobre a otimização.** Todos os modelos já estavam bem acima da meta antes de ajustar nada. O Grid Search serve mais para confirmar as configurações boas e tirar a dúvida de que dava para fazer melhor mexendo nos parâmetros. Os ganhos absolutos são pequenos, e isso é esperado num dataset limpo e bem separável: quando o sinal nos dados é forte, o teto de desempenho é atingido cedo. Numa situação real, com esse diagnóstico em mãos, o esforço renderia mais coletando dados de casos difíceis do que refinando hiperparâmetro.

<a id='5'></a>
## 5. Interpretação dos Resultados (CRISP-DM, Fase 5)

Acurácia alta sem entender o porquê é meio caminho andado. Nesta seção abro a caixa-preta por dois ângulos diferentes: a importância de features do Random Forest (quais medidas mais reduzem a incerteza) e os coeficientes da Regressão Logística (em que direção cada medida empurra cada classe). Quando essas duas leituras, vindas de modelos bem diferentes, apontam para o mesmo lado, a explicação ganha credibilidade.

In [ ]:
# Importância das features no Random Forest otimizado
best_rf = gs_rf.best_estimator_
importances = best_rf.feature_importances_
feat_names_pt = ['Área', 'Perímetro', 'Compacidade', 'Comp. Núcleo',
                 'Larg. Núcleo', 'Assimetria', 'Comp. Sulco']

feat_imp = pd.Series(importances, index=feat_names_pt).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors_fi = ['#d32f2f' if v == feat_imp.max() else '#42A5F5' for v in feat_imp.values]
feat_imp.plot.barh(ax=ax, color=colors_fi)
ax.set_title('Importância das Features: Random Forest Otimizado (Gini)')
ax.set_xlabel('Importância média (redução de impureza)')

for i, (v, name) in enumerate(zip(feat_imp.values, feat_imp.index)):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('seeds_importancia_features.png', bbox_inches='tight')
plt.show()

print('\nRanking de importância:')
print(feat_imp.sort_values(ascending=False).to_string())

In [ ]:
# Coeficientes da Regressão Logística (interpretabilidade linear)
coef_df = pd.DataFrame(
    lr_clf.coef_,
    columns=feat_names_pt,
    index=le.classes_
)
print('=== Coeficientes da Regressão Logística ===')
print('(valores positivos = feature aumenta probabilidade dessa classe)')
coef_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (classe, color) in zip(axes, zip(le.classes_, PALETTE)):
    coefs = coef_df.loc[classe].sort_values()
    bar_colors = ['#d32f2f' if c > 0 else '#1565C0' for c in coefs]
    coefs.plot.barh(ax=ax, color=bar_colors)
    ax.set_title(f'Coeficientes da Classe: {classe}', fontsize=10)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Coeficiente')

fig.suptitle('Regressão Logística: Coeficientes por Classe\n(vermelho = positivo, azul = negativo)',
             fontsize=12)
plt.tight_layout()
plt.savefig('seeds_coeficientes_lr.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== RESUMO FINAL DO PROJETO ===')
print()

all_models_summary = {}
for nome, vals in results.items():
    all_models_summary[nome] = vals['Acurácia']

# Inclui modelos otimizados
all_models_summary['KNN (Grid Search otimizado)']  = knn_opt_acc
all_models_summary['SVM (Grid Search otimizado)']  = svm_opt_acc
all_models_summary['RF  (Grid Search otimizado)']  = rf_opt_acc

best_name = max(all_models_summary, key=all_models_summary.get)
best_acc  = all_models_summary[best_name]

for nome, acc in sorted(all_models_summary.items(), key=lambda x: -x[1]):
    mark = '  <<< MELHOR MODELO' if nome == best_name else ''
    print(f'  {nome:42s} Acurácia: {acc:.4f} ({acc:.1%}){mark}')

print()
print(f'Meta de negócio (>=90%): {"ATINGIDA" if best_acc >= 0.9 else "NÃO ATINGIDA"}')
print(f'Melhor modelo: {best_name} com acurácia {best_acc:.4f} ({best_acc:.1%})')

### 5.1 Interpretação Detalhada dos Resultados

#### Importância das Features (Random Forest)

A importância por redução de impureza (Gini) mostra a hierarquia que a análise exploratória já vinha desenhando:

1. **Comprimento do núcleo** e **área** ficam no topo. São medidas que capturam tamanho de forma direta, e o tamanho é o que mais separa Rosa (grande) de Canadian (pequeno).
2. **Comprimento do sulco** também pesa bastante, o que é coerente com a alta correlação dele com o comprimento do núcleo: as duas medidas contam parte da mesma história.
3. **Compacidade** e **assimetria** aparecem mais embaixo quando olhadas isoladamente, mas não são descartáveis. É justamente nelas que o modelo se apoia para separar Kama de Canadian, que o tamanho sozinho não resolve.

#### Coeficientes da Regressão Logística

- Para **Rosa**, os coeficientes de área e perímetro são positivos: grão grande puxa para Rosa.
- Para **Canadian**, os mesmos coeficientes de tamanho são negativos: grão pequeno puxa para Canadian.
- Para **Kama**, o perfil é intermediário, e quem faz a diferença são a compacidade e a assimetria.

Repare que essa leitura bate com a importância de features do Random Forest. Dois modelos de mecânica completamente diferente concordam sobre o que separa as variedades, e isso é o melhor indício de que a explicação não é artefato de um algoritmo, é o que está nos dados.

#### Os erros que sobram

O punhado de erros que persiste em todos os modelos cai sempre entre **Kama e Canadian**. São grãos de Kama um pouco menores que a média da variedade, ou de Canadian um pouco maiores, que acabam na zona de sobreposição que vimos no gráfico de área contra perímetro. Para apertar esses casos, dois caminhos fariam mais sentido que trocar de algoritmo:

- Coletar mais amostras justamente nos extremos dessa fronteira (Kama pequeno e Canadian grande), para o modelo aprender melhor onde fica o limite.
- Acrescentar medidas que este dataset não tem, como textura da superfície do grão, que carregariam informação além da geometria.

### 5.2 Conclusões Gerais

1. **A automação é viável.** Todos os modelos passaram de 90% de acurácia. Um sistema baseado em SVM ou Random Forest otimizados pode entrar em operação numa cooperativa com confiabilidade alta o suficiente para substituir a triagem manual na maior parte dos casos.

2. **Modelo recomendado para produção: SVM com kernel RBF otimizado.** Ele combina a melhor acurácia com robustez e tempo de resposta rápido para os lotes pequenos típicos de cooperativas. Random Forest seria a alternativa natural se a equipe valorizar mais a interpretabilidade da importância de features.

3. **O dataset facilitou o trabalho.** A ausência de nulos e o balanceamento perfeito tornaram o pipeline mais curto. Em dados reais de campo, etapas de imputação, tratamento de ruído e possivelmente reamostragem voltariam a ser necessárias, e provavelmente seriam a parte mais trabalhosa.

4. **A otimização teve impacto modesto, e isso é uma resposta.** O Grid Search confirmou boas configurações sem mudar muito o resultado, sinal de que os modelos padrão já davam conta. O retorno marginal de mexer em hiperparâmetro aqui é baixo: o investimento mais produtivo seria em mais dados nos casos difíceis.

5. **As medidas de tamanho dominam.** Área, perímetro e comprimento do núcleo bastam para uma triagem inicial de alta precisão. Compacidade e assimetria entram como desempate, importantes principalmente nos casos ambíguos entre Kama e Canadian.

### 5.3 Próximos Passos (CRISP-DM, Fase 6: Implantação)

- **Empacotamento.** Juntar o `StandardScaler` e o SVM otimizado num único `Pipeline` do scikit-learn e exportar com `joblib`, para que a previsão em produção use exatamente a mesma transformação do treino, sem risco de divergência.
- **API.** Expor um endpoint simples (FastAPI ou Flask) que recebe as sete medidas e devolve a variedade junto com a probabilidade de cada classe, para que o sistema da cooperativa consuma sem conhecer os detalhes do modelo.
- **Interface de campo.** Um app ou até uma planilha que envie os dados ao classificador, pensando em quem realmente vai operar a ferramenta no dia a dia.
- **Monitoramento.** Acompanhar a acurácia ao longo do tempo para detectar drift, ou seja, lotes de safras ou regiões novas cujas características fujam do padrão com que o modelo foi treinado.
- **Expansão.** Incorporar dados de outras safras e regiões aos poucos, para o modelo deixar de valer só para esta amostra e ganhar generalidade.

In [ ]:
# Pipeline final recomendado para produção
from sklearn.pipeline import Pipeline
import joblib

best_svm_params = gs_svm.best_params_
pipeline_final = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(**best_svm_params, random_state=SEED, probability=True))
])

# Treino no dataset completo (todos os 210 registros)
pipeline_final.fit(X, y_enc)

# Exporta modelo
joblib.dump(pipeline_final, 'seeds_classifier_svm.joblib')
joblib.dump(le,             'seeds_label_encoder.joblib')

print('Pipeline SVM exportado: seeds_classifier_svm.joblib')
print('Label encoder exportado: seeds_label_encoder.joblib')
print()

# Demonstração de uso
amostra_nova = np.array([[15.26, 14.84, 0.871, 5.763, 3.312, 2.221, 5.220]])
predicao = pipeline_final.predict(amostra_nova)
prob      = pipeline_final.predict_proba(amostra_nova)

print(f'Amostra de teste: {amostra_nova[0]}')
print(f'Variedade predita: {le.inverse_transform(predicao)[0]}')
for i, (classe, p) in enumerate(zip(le.classes_, prob[0])):
    print(f'  P({classe}) = {p:.4f}')